In [ ]:
import sys
from pathlib import Path

# Dynamically resolve the project root and add it to the system path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Standardized directories for notebook usage
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from src.preprocessing import load_data, engineer_features, add_holidays, add_crisis_flags, apply_cyclical_encoding, fill_missing_dates, add_lag_features, split_datasets

import warnings
warnings.filterwarnings('ignore')

print("=== TEST MASSIF : APPROCHE RÉCURSIVE J+7 SUR JUIN 2026 ===\n")

# --- 1. CHARGEMENT ET PRÉPARATION DES DONNÉES HISTORIQUES ---
print("Chargement de l'historique (Master)...")
data_hist = load_data('data/Master.xlsx')
data_hist = engineer_features(data_hist)
data_hist[['Event', 'is_chol_hamoed', 'is_holiday_eve']] = data_hist['Date'].dt.date.apply(add_holidays)
data_hist = add_crisis_flags(data_hist)
data_hist = apply_cyclical_encoding(data_hist)
data_hist = fill_missing_dates(data_hist)
data_hist = add_lag_features(data_hist)

# --- 2. CHARGEMENT ET PRÉPARATION DES DONNÉES FUTURES (JUIN 2026) ---
print("Chargement des données de test (Juin 2026)...")
df_future = pd.read_excel('data/OCC DJ DT june.xlsx')
rename_map = {
    'מלון': 'Hotel',
    'תאריך הזמנה': 'Date',
    'חדרים למכירה': 'Total_Rooms',
    'חדרים תפוסים': 'Occupied_Rooms',
    'אחוז תפוסה': 'Occupancy_Rate',
    'אחוז ישראלים': 'Percentage_of_Israelis',
    'אחוז תיירים': 'Percentage_of_Tourists'
}
df_future = df_future.rename(columns=rename_map)
df_future['Hotel_ID'] = df_future['Hotel'].replace({'Dan Tel Aviv': 'DT', 'Dan Jerusalem': 'DJ', 'DJ': 'DJ'})
df_future['Date'] = pd.to_datetime(df_future['Date'])

df_future = engineer_features(df_future)
df_future[['Event', 'is_chol_hamoed', 'is_holiday_eve']] = df_future['Date'].dt.date.apply(add_holidays)
df_future = add_crisis_flags(df_future)
df_future = apply_cyclical_encoding(df_future)
df_future = add_lag_features(df_future)

# --- 3. CHARGEMENT DES MODÈLES ET SCALER ---
print("Chargement des modèles...")
model_dt = joblib.load('model_dt.pkl')
model_dj = joblib.load('model_dj.pkl')

expected_features_dt = model_dt.feature_names_in_

# Reconstitution du Scaler et des colonnes pour le DJ
train_pure, val_pure = split_datasets(data_hist)
full_dj_history = pd.concat([train_pure[train_pure['Hotel_ID'] == 'DJ'], val_pure[val_pure['Hotel_ID'] == 'DJ']])
features_base = ['DayWeek', 'Is_Weekend', 'sin_doy', 'cos_doy', 'Event', 'is_chol_hamoed', 'is_holiday_eve', 'Lag_1d', 'Lag_7d', 'Rolling_Mean_7d', 'Rolling_Mean_14d']

X_history_dj = pd.get_dummies(full_dj_history[features_base], columns=['Event'], drop_first=True)
expected_features_dj = X_history_dj.columns 
scaler_dj = StandardScaler().fit(X_history_dj)


# --- 4. FONCTION DE SIMULATION RÉCURSIVE ---
def run_recursive_simulation(start_date_str, hotel_id, model, expected_features, df_hist, df_fut, scaler=None):
    start_date = pd.to_datetime(start_date_str)
    target_date = start_date + pd.DateOffset(days=7)
    
    if target_date not in df_fut[df_fut['Hotel_ID']==hotel_id]['Date'].values:
        return None, None
        
    df_combined = pd.concat([df_hist[df_hist['Hotel_ID']==hotel_id], df_fut[df_fut['Hotel_ID']==hotel_id]])
    df_combined = df_combined.drop_duplicates(subset=['Date']).sort_values('Date')
    
    past_rates = df_combined[df_combined['Date'] <= start_date]['Target_Rate'].tolist()
    current_date = start_date
    
    for i in range(1, 8):
        current_date += pd.DateOffset(days=1)
        row = df_fut[(df_fut['Date'] == current_date) & (df_fut['Hotel_ID'] == hotel_id)].copy()
        if row.empty: return None, None
            
        row['Lag_1d'] = past_rates[-1]
        row['Lag_7d'] = past_rates[-7]
        row['Rolling_Mean_7d'] = np.mean(past_rates[-7:])
        row['Rolling_Mean_14d'] = np.mean(past_rates[-14:])
        
        X_infer = pd.get_dummies(row[features_base], columns=['Event'], drop_first=True)
        X_infer = X_infer.reindex(columns=expected_features, fill_value=0)
        
        if scaler:
            X_infer = scaler.transform(X_infer)
            
        pred = np.clip(model.predict(X_infer)[0], 0.0, 1.0)
        past_rates.append(pred)
        
    actual_target = df_fut[(df_fut['Date'] == target_date) & (df_fut['Hotel_ID'] == hotel_id)]['Target_Rate'].values[0]
    return past_rates[-1], actual_target


# --- 5. EXÉCUTION DES TESTS MASSIFS ---
print("\nSimulation des dérives en cours...\n")
# On teste un départ tous les 2 jours de juin
start_dates_to_test = ['2026-06-05', '2026-06-07', '2026-06-09', '2026-06-11', '2026-06-13', '2026-06-15', '2026-06-17', '2026-06-19']
results = []

for start_date in start_dates_to_test:
    target = (pd.to_datetime(start_date) + pd.DateOffset(days=7)).strftime('%Y-%m-%d')
    
    pred_dt, act_dt = run_recursive_simulation(start_date, 'DT', model_dt, expected_features_dt, data_hist, df_future)
    err_dt = abs(pred_dt - act_dt) * 100 if pred_dt is not None else None
    
    pred_dj, act_dj = run_recursive_simulation(start_date, 'DJ', model_dj, expected_features_dj, data_hist, df_future, scaler_dj)
    err_dj = abs(pred_dj - act_dj) * 100 if pred_dj is not None else None
    
    if err_dt is not None and err_dj is not None:
        results.append({
            'Start_Date': start_date,
            'Target_Date': target,
            'DT_Pred(%)': round(pred_dt*100, 1),
            'DT_Actual(%)': round(act_dt*100, 1),
            'DT_Error(pts)': round(err_dt, 1),
            'DJ_Pred(%)': round(pred_dj*100, 1),
            'DJ_Actual(%)': round(act_dj*100, 1),
            'DJ_Error(pts)': round(err_dj, 1)
        })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

print("\n--- BILAN GLOBAL DE L'APPROCHE RÉCURSIVE J+7 ---")
print(f"MAE Moyenne Dan Tel Aviv : {df_results['DT_Error(pts)'].mean():.2f} points d'erreur")
print(f"MAE Moyenne Dan Jérusalem : {df_results['DJ_Error(pts)'].mean():.2f} points d'erreur")

=== TEST MASSIF : APPROCHE RÉCURSIVE J+7 SUR JUIN 2026 ===

Chargement de l'historique (Master)...
Chargement des données de test (Juin 2026)...
Chargement des modèles...

Simulation des dérives en cours...

Start_Date Target_Date  DT_Pred(%)  DT_Actual(%)  DT_Error(pts)  DJ_Pred(%)  DJ_Actual(%)  DJ_Error(pts)
2026-06-05  2026-06-12        63.1          94.5           31.5        69.4          92.7           23.3
2026-06-07  2026-06-14        48.7          29.4           19.4        26.1          26.1            0.0
2026-06-09  2026-06-16        56.3          41.0           15.4        26.8          11.7           15.2

--- BILAN GLOBAL DE L'APPROCHE RÉCURSIVE J+7 ---
MAE Moyenne Dan Tel Aviv : 22.10 points d'erreur
MAE Moyenne Dan Jérusalem : 12.83 points d'erreur
